## RAG 챌린지

In [1]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores import Chroma
from langchain.storage import LocalFileStore
from langchain.memory import ConversationBufferMemory
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.schema.runnable import RunnablePassthrough

llm = ChatOpenAI(
    temperature=0.1,
)

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

loader = TextLoader("./files/document.txt")

docs = loader.load_and_split(
    text_splitter=splitter,
)

cache_dir = LocalFileStore("./.cache/")

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings,
    cache_dir,
)

vectorstore = Chroma.from_documents(
    docs,
    cached_embeddings,
)

retriever = vectorstore.as_retriever()

memory = ConversationBufferMemory(
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a helpful AI assistant.

        Answer the user's question using only the following context.
        If you don't know the answer based on the context, say you don't know.

        Context:
        {context}
        """
    ),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

def load_memory(_):
    return memory.load_memory_variables({})["history"]


def format_docs(docs):
    return "\n\n".join(
        document.page_content for document in docs
    )

def retrieve_docs(inputs):
    docs = retriever.invoke(inputs["question"])
    return format_docs(docs)

chain = (
    RunnablePassthrough.assign(
        context=retrieve_docs,
        history=load_memory,
    )
    | prompt
    | llm
)

def invoke_chain(question):
    result = chain.invoke({
        "question": question,
    })

    memory.save_context(
        {"input": question},
        {"output": result.content},
    )

    print(result.content)

In [2]:
invoke_chain("Is Aaronson guilty?")

Yes, according to the context, Jones, Aaronson, and Rutherford were guilty of the crimes they were charged with.


In [3]:
invoke_chain("What message did he write in the table?")

He wrote "FREEDOM IS SLAVERY" and then "TWO AND TWO MAKE FIVE" on the table.


In [4]:
invoke_chain("Who is Julia?")

Julia is a character in the context who was involved with Winston, the main character, in acts of rebellion against the Party.
